# Text Preprocessing for IMDB Sentiment Analysis

This notebook implements comprehensive text preprocessing pipeline for the IMDB movie reviews dataset.

**Preprocessing Steps**:
1. HTML tag removal
2. Text normalization (lowercase, special characters)
3. Tokenization
4. Stop word removal (optional)
5. Stemming/Lemmatization (optional)
6. Feature extraction (TF-IDF, embeddings)
7. Train/validation/test split with stratification

## 1. Setup and Data Loading

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import re
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# NLP libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Scikit-learn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("Libraries imported successfully!")

In [ ]:
# Import data loader
import sys
sys.path.append('..')
from text_classification.data_loader import IMDBDataLoader

# Load dataset
loader = IMDBDataLoader()
train_df, test_df = loader.load_dataset()

print(f"Training set: {train_df.shape}")
print(f"Test set: {test_df.shape}")
print(f"\nSample review (raw):")
print(train_df['text'].iloc[0][:500])

## 2. Text Cleaning Functions

In [ ]:
def remove_html_tags(text):
    """Remove HTML tags from text"""
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)

def remove_urls(text):
    """Remove URLs from text"""
    return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

def remove_special_chars(text, keep_apostrophe=True):
    """Remove special characters, keep only letters and spaces"""
    if keep_apostrophe:
        pattern = r'[^a-zA-Z\s\']'
    else:
        pattern = r'[^a-zA-Z\s]'
    return re.sub(pattern, '', text)

def remove_extra_spaces(text):
    """Remove extra whitespace"""
    return ' '.join(text.split())

def expand_contractions(text):
    """Expand common contractions"""
    contractions = {
        "n't": " not",
        "'re": " are",
        "'s": " is",
        "'d": " would",
        "'ll": " will",
        "'ve": " have",
        "'m": " am"
    }
    for contraction, expansion in contractions.items():
        text = text.replace(contraction, expansion)
    return text

# Test cleaning functions
sample_text = "<br /><br />This movie isn't very good! It's terrible. www.example.com"
print("Original:", sample_text)
print("Remove HTML:", remove_html_tags(sample_text))
print("Remove URLs:", remove_urls(sample_text))
print("Expand contractions:", expand_contractions(sample_text))
print("Remove special chars:", remove_special_chars(sample_text))

## 3. Complete Preprocessing Pipeline

In [ ]:
class TextPreprocessor:
    """Comprehensive text preprocessing pipeline"""
    
    def __init__(self, 
                 lowercase=True,
                 remove_stopwords=False,
                 use_stemming=False,
                 use_lemmatization=False):
        """
        Initialize preprocessor with configuration
        
        Args:
            lowercase: Convert text to lowercase
            remove_stopwords: Remove common stopwords
            use_stemming: Apply Porter stemming
            use_lemmatization: Apply lemmatization
        """
        self.lowercase = lowercase
        self.remove_stopwords = remove_stopwords
        self.use_stemming = use_stemming
        self.use_lemmatization = use_lemmatization
        
        # Initialize tools
        self.stop_words = set(stopwords.words('english')) if remove_stopwords else set()
        self.stemmer = PorterStemmer() if use_stemming else None
        self.lemmatizer = WordNetLemmatizer() if use_lemmatization else None
        
    def clean_text(self, text):
        """Apply all cleaning steps to text"""
        # Remove HTML tags
        text = remove_html_tags(text)
        
        # Remove URLs
        text = remove_urls(text)
        
        # Expand contractions
        text = expand_contractions(text)
        
        # Convert to lowercase
        if self.lowercase:
            text = text.lower()
        
        # Remove special characters
        text = remove_special_chars(text, keep_apostrophe=False)
        
        # Remove extra spaces
        text = remove_extra_spaces(text)
        
        return text
    
    def tokenize_and_process(self, text):
        """Tokenize and apply stemming/lemmatization"""
        # Tokenize
        tokens = text.split()  # Simple whitespace tokenization
        
        # Remove stopwords
        if self.remove_stopwords:
            tokens = [word for word in tokens if word not in self.stop_words]
        
        # Apply stemming
        if self.use_stemming and self.stemmer:
            tokens = [self.stemmer.stem(word) for word in tokens]
        
        # Apply lemmatization
        if self.use_lemmatization and self.lemmatizer:
            tokens = [self.lemmatizer.lemmatize(word) for word in tokens]
        
        return ' '.join(tokens)
    
    def preprocess(self, text):
        """Complete preprocessing pipeline"""
        text = self.clean_text(text)
        text = self.tokenize_and_process(text)
        return text
    
    def preprocess_dataframe(self, df, text_column='text'):
        """Preprocess all texts in a DataFrame"""
        df = df.copy()
        df['cleaned_text'] = df[text_column].apply(self.preprocess)
        return df

print("TextPreprocessor class created successfully!")

## 4. Test Different Preprocessing Configurations

In [ ]:
# Test sample text with different configurations
sample = train_df['text'].iloc[0]

print("ORIGINAL TEXT:")
print(sample[:300])
print("\n" + "="*80 + "\n")

# Configuration 1: Basic cleaning
preprocessor1 = TextPreprocessor(lowercase=True, remove_stopwords=False)
cleaned1 = preprocessor1.preprocess(sample)
print("CONFIG 1 - Basic cleaning (lowercase, no stopword removal):")
print(cleaned1[:300])
print(f"Word count: {len(cleaned1.split())}")
print("\n" + "="*80 + "\n")

# Configuration 2: With stopword removal
preprocessor2 = TextPreprocessor(lowercase=True, remove_stopwords=True)
cleaned2 = preprocessor2.preprocess(sample)
print("CONFIG 2 - With stopword removal:")
print(cleaned2[:300])
print(f"Word count: {len(cleaned2.split())}")
print("\n" + "="*80 + "\n")

# Configuration 3: With stemming
preprocessor3 = TextPreprocessor(lowercase=True, remove_stopwords=False, use_stemming=True)
cleaned3 = preprocessor3.preprocess(sample)
print("CONFIG 3 - With stemming:")
print(cleaned3[:300])
print(f"Word count: {len(cleaned3.split())}")

## 5. Apply Preprocessing to Full Dataset

In [ ]:
# Choose preprocessing configuration
# For most ML models, basic cleaning without stopword removal works well
preprocessor = TextPreprocessor(
    lowercase=True,
    remove_stopwords=False,  # Keep for now, can experiment later
    use_stemming=False,      # Can reduce vocabulary but may lose semantic meaning
    use_lemmatization=False  # More accurate but slower than stemming
)

print("Preprocessing training data...")
train_processed = preprocessor.preprocess_dataframe(train_df)

print("Preprocessing test data...")
test_processed = preprocessor.preprocess_dataframe(test_df)

print("\nPreprocessing complete!")
print(f"Train shape: {train_processed.shape}")
print(f"Test shape: {test_processed.shape}")

# Display sample
print("\nSample preprocessed review:")
print(train_processed['cleaned_text'].iloc[0][:500])

## 6. Create Train/Validation Split

In [ ]:
# Split training data into train and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_processed['cleaned_text'],
    train_processed['label'],
    test_size=0.2,
    random_state=42,
    stratify=train_processed['label']  # Ensure balanced splits
)

test_texts = test_processed['cleaned_text']
test_labels = test_processed['label']

print("Data Split Summary:")
print(f"  Training: {len(train_texts):,} samples ({len(train_texts)/(len(train_texts)+len(val_texts))*100:.1f}%)")
print(f"  Validation: {len(val_texts):,} samples ({len(val_texts)/(len(train_texts)+len(val_texts))*100:.1f}%)")
print(f"  Test: {len(test_texts):,} samples")

print("\nClass distribution:")
print(f"  Train - Positive: {(train_labels==1).sum()}, Negative: {(train_labels==0).sum()}")
print(f"  Val - Positive: {(val_labels==1).sum()}, Negative: {(val_labels==0).sum()}")
print(f"  Test - Positive: {(test_labels==1).sum()}, Negative: {(test_labels==0).sum()}")

## 7. Feature Extraction - TF-IDF

In [ ]:
# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,      # Limit vocabulary to top 10k words
    min_df=5,                # Word must appear in at least 5 documents
    max_df=0.7,              # Word must appear in less than 70% of documents
    ngram_range=(1, 2),      # Use unigrams and bigrams
)

# Fit on training data and transform all splits
print("Extracting TF-IDF features...")
X_train_tfidf = tfidf_vectorizer.fit_transform(train_texts)
X_val_tfidf = tfidf_vectorizer.transform(val_texts)
X_test_tfidf = tfidf_vectorizer.transform(test_texts)

print(f"\nTF-IDF Feature Matrix Shapes:")
print(f"  Train: {X_train_tfidf.shape}")
print(f"  Validation: {X_val_tfidf.shape}")
print(f"  Test: {X_test_tfidf.shape}")
print(f"\nVocabulary size: {len(tfidf_vectorizer.vocabulary_):,}")

# Show top features
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"\nSample features: {list(feature_names[:20])}")

## 8. Feature Extraction - Count Vectorizer (BoW)

In [ ]:
# Create Count Vectorizer (Bag of Words)
count_vectorizer = CountVectorizer(
    max_features=10000,
    min_df=5,
    max_df=0.7,
    ngram_range=(1, 2)
)

# Fit and transform
print("Extracting Bag-of-Words features...")
X_train_bow = count_vectorizer.fit_transform(train_texts)
X_val_bow = count_vectorizer.transform(val_texts)
X_test_bow = count_vectorizer.transform(test_texts)

print(f"\nBoW Feature Matrix Shapes:")
print(f"  Train: {X_train_bow.shape}")
print(f"  Validation: {X_val_bow.shape}")
print(f"  Test: {X_test_bow.shape}")

## 9. Save Preprocessed Data

In [ ]:
# Create processed data directory
processed_dir = Path('../datasets/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

# Save preprocessed text data
train_processed.to_csv(processed_dir / 'train_preprocessed.csv', index=False)
test_processed.to_csv(processed_dir / 'test_preprocessed.csv', index=False)

print("Saved preprocessed text data")

# Save train/val/test splits as pickle
import pickle

splits_data = {
    'train_texts': train_texts,
    'val_texts': val_texts,
    'test_texts': test_texts,
    'train_labels': train_labels,
    'val_labels': val_labels,
    'test_labels': test_labels
}

with open(processed_dir / 'data_splits.pkl', 'wb') as f:
    pickle.dump(splits_data, f)

print("Saved data splits")

# Save TF-IDF features and vectorizer
tfidf_data = {
    'vectorizer': tfidf_vectorizer,
    'X_train': X_train_tfidf,
    'X_val': X_val_tfidf,
    'X_test': X_test_tfidf,
    'y_train': train_labels.values,
    'y_val': val_labels.values,
    'y_test': test_labels.values
}

with open(processed_dir / 'tfidf_features.pkl', 'wb') as f:
    pickle.dump(tfidf_data, f)

print("Saved TF-IDF features")

# Save BoW features and vectorizer
bow_data = {
    'vectorizer': count_vectorizer,
    'X_train': X_train_bow,
    'X_val': X_val_bow,
    'X_test': X_test_bow,
    'y_train': train_labels.values,
    'y_val': val_labels.values,
    'y_test': test_labels.values
}

with open(processed_dir / 'bow_features.pkl', 'wb') as f:
    pickle.dump(bow_data, f)

print("Saved BoW features")

# Save preprocessor configuration
with open(processed_dir / 'preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

print("Saved preprocessor")

print(f"\nAll preprocessed data saved to: {processed_dir.absolute()}")

## 10. Summary and Next Steps

In [ ]:
print("="*80)
print("PREPROCESSING SUMMARY")
print("="*80)

print("\n1. CLEANING STEPS APPLIED:")
print("   - HTML tag removal")
print("   - URL removal")
print("   - Contraction expansion")
print("   - Lowercase conversion")
print("   - Special character removal")
print("   - Extra whitespace removal")

print("\n2. DATA SPLITS:")
print(f"   - Training: {len(train_texts):,} samples (64% of original train)")
print(f"   - Validation: {len(val_texts):,} samples (16% of original train)")
print(f"   - Test: {len(test_texts):,} samples (20% of total data)")

print("\n3. FEATURE EXTRACTION:")
print(f"   - TF-IDF: {X_train_tfidf.shape[1]:,} features")
print(f"   - Bag-of-Words: {X_train_bow.shape[1]:,} features")
print(f"   - N-grams: Unigrams + Bigrams")

print("\n4. SAVED FILES:")
print(f"   - Preprocessed CSVs: train_preprocessed.csv, test_preprocessed.csv")
print(f"   - Data splits: data_splits.pkl")
print(f"   - TF-IDF features: tfidf_features.pkl")
print(f"   - BoW features: bow_features.pkl")
print(f"   - Preprocessor: preprocessor.pkl")

print("\n5. NEXT STEPS - MODEL TRAINING:")
print("   a. Baseline Models:")
print("      - Naive Bayes (MultinomialNB)")
print("      - Logistic Regression")
print("   b. Advanced Models:")
print("      - Support Vector Machine (SVM)")
print("      - Random Forest")
print("      - XGBoost")
print("   c. Deep Learning Models:")
print("      - LSTM with word embeddings")
print("      - CNN for text classification")
print("      - BERT-based models")
print("   d. Hyperparameter Tuning:")
print("      - Grid search or random search")
print("      - Cross-validation")
print("   e. Model Evaluation:")
print("      - Accuracy, precision, recall, F1-score")
print("      - Confusion matrix")
print("      - Comparison with benchmarks (88-95%)")

print("\n" + "="*80)
print("Preprocessing pipeline complete! Ready for model training.")
print("="*80)